In [1]:
import os.path as op 
import os
from glob import glob
import pickle
import pingouin as pg
import seaborn as sns
import pandas as pd
import numpy as np

In [2]:
qa_df = pd.read_csv('/global/homes/m/mphagen/functional-connectivity/connectome-comparison/results/2026_05_01_absolute_movement.csv', 
                    index_col=0).sort_index()
qa_df.index = [f'sub-{i}' for i in qa_df.index] 
print(qa_df.shape, qa_df.columns)

(1113, 4) Index(['RL_1', 'RL_2', 'LR_2', 'LR_1'], dtype='object')


In [3]:
#adopt future option 
pd.set_option('future.no_silent_downcasting', True)

unrestricted_df = pd.read_csv('../../../data/unrestricted_mphagen_1_27_2022_20_50_7.csv')
unrestricted_df.index = [f'sub-{i}' for i in unrestricted_df['Subject']]
unrestricted_df[['Gender']] = unrestricted_df[['Gender']].replace({'M':0, 'F':1}).infer_objects(copy=False)

In [4]:
restricted_df = pd.read_csv('../../../data/RESTRICTED_arokem_1_31_2022_23_26_45.csv')
restricted_df.index = [f'sub-{i}' for i in restricted_df['Subject']]

In [5]:
covariates = ['Gender', 'Age_in_Yrs']
covariate_df = unrestricted_df[['Gender']].join(restricted_df[['Age_in_Yrs']]) 

In [6]:
def make_fc_df(results_dict, ses): 
    fc_df = pd.DataFrame(columns=results_dict.keys()) 
    
    for key in results_dict.keys(): 
        fc_df[key] = results_dict[key][f'ses-{ses}'].flatten() 
    fc_df = fc_df.T
    # if model == 'correlation': 
    #      fc_df
    # fc_df = fc_df.drop(fc_df.columns[fc_df.nunique() <= 1],axis=1) 
    return fc_df

In [7]:
def make_qcfc_df(fc_df, cov_df, qa_df, ses): 
    qa_df[[f'RL_{ses}', f'LR_{ses}']]
    connectomes = fc_df.join(covariate_df) 
    rms_df = qa_df[[f'RL_{ses}', f'LR_{ses}']].mean(axis=1).rename('rms')
    connectomes = connectomes.join(rms_df)
    return connectomes

In [8]:
def qcfc(qcfc_df, covariates, edge_range): 
    result_df = pd.DataFrame(index=edge_range, columns=['r',  'p-val'])
    result_df.index = [f'edge-{i}' for i in result_df.index]

    results_list = {}
    for edge_id in edge_range:
    # QC-FC
        metric = pg.partial_corr(data=qcfc_df, 
                    x=edge_id, 
                    y='rms', 
                    covar=covariates)

        result_df.loc[f'edge-{edge_id}'] = metric.loc['pearson', ['r', 'p-val']]
    return result_df

In [9]:
def run_qcfc(model, proc_type, ses, date_str='2026-05-22'): 
    result_dict = {} 
    with open(glob(op.join(results_path, proc_type, f'*{date_str}*{model}*relmat.pkl'))[0], 'rb') as l:
        connectome_dict = pickle.load(l)

    fc_df = make_fc_df(connectome_dict, ses=ses)
    edge_range = fc_df.columns

    qcfc_df = make_qcfc_df(fc_df, covariate_df, qa_df, ses=ses)
    ## add resampling here
    result_dict[f'{model}_{proc_type}_ses-{ses}'] = qcfc(qcfc_df, covariates, edge_range)
    return result_dict

In [10]:
qcfc_dict = {} 

In [11]:
results_path = '/global/homes/m/mphagen/functional-connectivity/connectome-comparison/results'


In [12]:
# model = 'lassoBIC'
# proc_type = 'MSMAll'
# with open(glob(op.join(results_path, proc_type, f'*12-10*{model}*relmat.pkl'))[0], 'rb') as l:
#         connectome_dict = pickle.load(l)

# fc_df = make_fc_df(connectome_dict)

In [12]:
import warnings
# Or ignore specific categories
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [14]:
# model='lassoBIC'

# proc_type = 'MSMAll'
# result_dict = {} 
# with open(glob(op.join(results_path, proc_type, f'*2026*{model}*relmat.pkl'))[0], 'rb') as l:
#         msmall_dict = pickle.load(l)

In [15]:
# proc_type = 'msmallFIX'
# result_dict = {} 
# with open(glob(op.join(results_path, proc_type, f'*2026*{model}*relmat.pkl'))[0], 'rb') as l:
#         msmall_fix_dict = pickle.load(l)

In [16]:
# msmall_f_1_fc_df = make_fc_df(msmall_fix_dict, ses=1)
# msmall_f_2_fc_df = make_fc_df(msmall_fix_dict, ses=2)


In [17]:
# msmall_1_fc_df = make_fc_df(msmall_dict, ses=1)
# msmall_2_fc_df = make_fc_df(msmall_fix_dict, ses=2)


In [18]:
# edge_range = msmall_1_fc_df.columns


In [19]:
# res_df = qcfc(qcfc_df, covariates, edge_range)

In [20]:
# res_df.loc[res_df['p-val'] <= .05].shape

In [14]:
date_str = '2026-05-22'

In [22]:
model = 'lassoBIC'
proc_type = 'MSMAll'
qcfc_dict.update(run_qcfc(model, proc_type, date_str='2026', ses='1')) 
qcfc_dict.update(run_qcfc(model, proc_type, date_str='2026', ses='2')) 

In [23]:
model = 'lassoBIC'
proc_type = 'MSMAll_FIX'
qcfc_dict.update(run_qcfc(model, proc_type, date_str='2026-05-22', ses='1')) 
qcfc_dict.update(run_qcfc(model, proc_type, date_str='2026-05-22', ses='2')) 

In [15]:
model = '-correlation'
proc_type = 'MSMAll_FIX'
qcfc_dict.update(run_qcfc(model, proc_type, date_str=date_str, ses='1')) 
qcfc_dict.update(run_qcfc(model, proc_type, date_str=date_str, ses='2')) 

In [25]:
# model = 'lassoBIC'
# proc_type = 'xcpdPNC'
# qcfc_dict.update(run_qcfc(model, proc_type, date_str='2026', ses='1')) 
# qcfc_dict.update(run_qcfc(model, proc_type, date_str='2026', ses='2'))

In [26]:
model = 'correlation'
proc_type = 'MSMAll'
qcfc_dict.update(run_qcfc(model, proc_type, '1')) 
qcfc_dict.update(run_qcfc(model, proc_type, '2')) 

IndexError: list index out of range

In [27]:
model = '-correlation'
proc_type = 'xcpd'
qcfc_dict.update(run_qcfc(model, proc_type, '1')) 
qcfc_dict.update(run_qcfc(model, proc_type, '2')) 

In [28]:
model = 'lassoBIC'
proc_type = 'xcpd'
qcfc_dict.update(run_qcfc(model, proc_type, '1')) 
qcfc_dict.update(run_qcfc(model, proc_type, '2')) 

In [ ]:
model = 'corr-lassoThresh'
proc_type = 'xcpd'
qcfc_dict.update(run_qcfc(model, proc_type, '1')) 
qcfc_dict.update(run_qcfc(model, proc_type, '2')) 

In [ ]:
model = 'corr-lassoThresh'
proc_type = 'MSMAll'
qcfc_dict.update(run_qcfc(model, proc_type, '1')) 
qcfc_dict.update(run_qcfc(model, proc_type, '2')) 

In [ ]:
import pickle

In [ ]:
# with open('qcfc_dict.pkl', 'wb') as file: 
#     pickle.dump(qcfc_dict) 

In [29]:
mask = np.triu(np.ones_like(np.ones((100,100)), dtype=bool), k=1) 

In [30]:
qcfc_dict.keys() 

dict_keys(['lassoBIC_MSMAll_ses-1', 'lassoBIC_MSMAll_ses-2', 'lassoBIC_MSMAll_FIX_ses-1', 'lassoBIC_MSMAll_FIX_ses-2', '-correlation_MSMAll_FIX_ses-1', '-correlation_MSMAll_FIX_ses-2', '-correlation_xcpd_ses-1', '-correlation_xcpd_ses-2', 'lassoBIC_xcpd_ses-1', 'lassoBIC_xcpd_ses-2'])

In [16]:
df = qcfc_dict['correlation_MSMAll_ses-1']
df = df.loc[mask.ravel()]

df.loc[df['p-val'] <= .05].shape

KeyError: 'correlation_MSMAll_ses-1'

In [ ]:
df = qcfc_dict['correlation_MSMAll_ses-2']
df = df.loc[mask.ravel()]

df.loc[df['p-val'] <= .05].shape

In [18]:
df = qcfc_dict['-correlation_MSMAll_FIX_ses-1']

df.loc[df['p-val'] <= .05].shape

(262, 2)

In [34]:
import matplotlib.pyplot as plt

In [20]:
df = qcfc_dict['-correlation_MSMAll_FIX_ses-2']

df.loc[df['p-val'] <= .05].shape

(3390, 2)

In [ ]:
plt.hist(df.loc[df['p-val'] <= .05]['r'])

In [ ]:
df = qcfc_dict['lassoBIC_MSMAll_FIX_ses-1']

df.loc[df['p-val'] <= .05].shape

In [ ]:
df = qcfc_dict['lassoBIC_MSMAll_FIX_ses-2']

df.loc[df['p-val'] <= .05].shape

In [ ]:
df = qcfc_dict['lassoBIC_MSMAll_ses-1']

df.loc[df['p-val'] <= .05].shape

In [ ]:
df = qcfc_dict['lassoBIC_MSMAll_ses-2']

df.loc[df['p-val'] <= .05].shape

In [ ]:
df = qcfc_dict['corr-lassoThresh_MSMAll_ses-1']
df = df.loc[mask.ravel()]

df.loc[df['p-val'] <= .05] 

In [ ]:
df = qcfc_dict['correlation_MSMAll_ses-1']
df = df.loc[mask.ravel()]

df.loc[df['p-val'] <= .05] 

In [ ]:
df = qcfc_dict['corr-lassoThresh_MSMAll_ses-2']
df = df.loc[mask.ravel()]

df.loc[df['p-val'] <= .05] 

In [ ]:
df = qcfc_dict['correlation_MSMAll_ses-2']
df = df.loc[mask.ravel()]

df.loc[df['p-val'] <= .05] 

In [ ]:
df = qcfc_dict['correlation_xcpd_ses-1']
df = df.loc[mask.ravel()]
df.loc[df['p-val'] <= .05] 

In [ ]:
df = qcfc_dict['corr-lassoThresh_xcpd_ses-1']
df = df.loc[mask.ravel()]
df.loc[df['p-val'] <= .05] 

In [ ]:
df = qcfc_dict['corr-lassoThresh_xcpd_ses-2']
df = df.loc[mask.ravel()]
df.loc[df['p-val'] <= .05] 

In [ ]:
df = qcfc_dict['correlation_xcpd_ses-2']
df = df.loc[mask.ravel()]
df.loc[df['p-val'] <= .05] 

In [ ]:
df = qcfc_dict['correlation_xcpd_ses-1']
df = df.loc[mask.ravel()]
df.loc[df['p-val'] <= .05] 

In [ ]:
df = qcfc_dict['lassoBIC_xcpd_ses-1']
df.loc[df['p-val'] <= .05] 